# 02b — Ingest real earnings-call transcripts (HuggingFace)

Swaps the transcript source from SEC 8-K press releases to **actual earnings
call transcripts** with per-speaker turns:
[`kurry/sp500_earnings_transcripts`](https://huggingface.co/datasets/kurry/sp500_earnings_transcripts)
— 33k+ calls, 685 companies, 2005–2025, MIT license.

What this notebook does:
1. Downloads the dataset (~1.8 GB, cached by HuggingFace outside OneDrive)
2. Splits each call into **prepared remarks vs Q&A using speaker turns**
   (reliable — no regex guessing)
3. Computes Q&A/evasiveness stats once at ingest (hedge rate, answer/question
   length ratio, analyst counts)
4. Stores everything **zlib-compressed** in one SQLite table
   (`transcripts_text`) — no 100k small files, ~4-5x smaller on disk

Downstream: `03_sentiment.ipynb` reads sections straight from this table.

In [ ]:
import sys, time
sys.path.insert(0, '..')
import pandas as pd
from src.config import DB_PATH
from src.transcripts_io import (
    normalize_turns, split_turns, flat_sections, make_row,
    open_db, write_batch, META_COLS,
)

HF_DATASET = 'kurry/sp500_earnings_transcripts'
BATCH_SIZE = 500      # rows per DB transaction (bigger = fewer fsyncs)
LOG_EVERY = 2500
MIN_YEAR = 2005       # ingest everything; 03 filters at read time

print(f'DB: {DB_PATH}')

In [ ]:
from datasets import load_dataset

t0 = time.time()
# Arrow-backed + memory-mapped: iterating does NOT load 1.8 GB into RAM.
ds = load_dataset(HF_DATASET, split='train')
# Drop columns we never read — skips their Arrow->Python decode per row.
ds = ds.select_columns(['symbol', 'year', 'quarter', 'date',
                        'structured_content', 'content'])
print(f'Loaded {len(ds):,} transcripts in {time.time()-t0:.0f}s')
r0 = ds[0]
print('Sample:', r0['symbol'], r0['year'], f"q{r0['quarter']}", str(r0['date'])[:10],
      f"| turns: {len(r0['structured_content'] or [])}")

In [ ]:
conn = open_db(DB_PATH)

n_ok = n_skip = n_flat = 0
batch = []
t0 = time.time()

for i, row in enumerate(ds):
    year, q = int(row['year']), int(row['quarter'])
    if year < MIN_YEAR or not (1 <= q <= 4):
        n_skip += 1
        continue

    turns = normalize_turns(row['structured_content'])
    if turns:
        sections = split_turns(turns)
    else:
        # No speaker structure — fall back to the flat content field
        text = (row['content'] or '').strip()
        if not text:
            n_skip += 1
            continue
        n_flat += 1
        sections = flat_sections(text)

    if not sections['full_text']:
        n_skip += 1
        continue

    batch.append(make_row(row['symbol'], f'q{q}', year,
                          str(row['date'])[:10], 'hf', sections))
    if len(batch) >= BATCH_SIZE:
        write_batch(conn, batch)
        n_ok += len(batch)
        batch = []

    if (i + 1) % LOG_EVERY == 0:
        el = time.time() - t0
        print(f'  [{i+1:,}/{len(ds):,}] ok={n_ok:,} skip={n_skip} '
              f'flat={n_flat} ({el:.0f}s, {(i+1)/el:.0f} rows/s)')

write_batch(conn, batch)
n_ok += len(batch)
print(f'\nDone in {time.time()-t0:.0f}s: {n_ok:,} ingested, '
      f'{n_skip} skipped, {n_flat} flat (no speaker turns)')

In [ ]:
# Sanity summary — metadata only, never touches the compressed BLOBs
meta = pd.read_sql(f"SELECT {', '.join(META_COLS)} FROM transcripts_text", conn)
conn.close()

print(f'Rows in transcripts_text: {len(meta):,}')
print(f'Tickers: {meta["ticker"].nunique()}')
print(f'Date range: {meta["pub_date"].min()} to {meta["pub_date"].max()}')
print(f'Q&A coverage: {meta["has_qa"].mean():.1%}  (vs ~40-50% with 8-K scraping)')
print(f'Median words/call: {meta["n_words_full"].median():,.0f}')
print(f'Median analyst questions: {meta["n_analyst_questions"].median():.0f}')
print(f'Median hedge rate: {meta["hedge_per_1k"].median():.1f} per 1k answer words')

from pathlib import Path
print(f'\nDB size on disk: {Path(DB_PATH).stat().st_size / 1e6:,.0f} MB')
print('\nCalls per year:')
print(meta.groupby('year').size().to_string())

In [ ]:
# ---- Keeping data current (post-HF quarters) via defeatbeta-api --------
# The HF dataset ends mid-2025. defeatbeta-api (free, no key, MIT) serves
# speaker-segmented transcripts from HuggingFace-hosted parquet, updated
# weekly — verified current through 2026 quarters.
#
# Incremental by construction: for each ticker we only fetch quarters whose
# report_date is newer than the ticker's latest stored pub_date (tickers
# not in the DB get their full history). Safe to interrupt and re-run.
#
# NOTE: `pip install defeatbeta-api` upgrades numpy/pandas beyond our pins;
# afterwards run `pip install numpy==1.26.4 pandas==2.2.0` to restore them
# (the library still works — its floor is metadata-only).
RUN_UPDATE = False

if RUN_UPDATE:
    from defeatbeta_api.data.ticker import Ticker
    from src.config import get_full_universe

    conn = open_db(DB_PATH)
    known = pd.read_sql(
        'SELECT ticker, quarter, year, pub_date FROM transcripts_text', conn)
    existing = set(known[['ticker', 'quarter', 'year']]
                   .itertuples(index=False, name=None))
    latest_pub = known.groupby('ticker')['pub_date'].max().to_dict()

    universe = get_full_universe(DB_PATH)
    n_new = n_skip = n_fail = 0
    t0 = time.time()

    for i, symbol in enumerate(universe):
        try:
            tr = Ticker(symbol).earning_call_transcripts()
            listing = tr.get_transcripts_list()
            # columns: symbol, fiscal_year, fiscal_quarter, report_date
        except Exception as e:
            n_fail += 1
            print(f'  {symbol}: listing failed ({type(e).__name__}: {e})')
            continue

        cutoff = latest_pub.get(symbol, '')  # '' -> take full history
        batch = []
        for r in listing.itertuples():
            year, q = int(r.fiscal_year), int(r.fiscal_quarter)
            pub_date = str(r.report_date)[:10]
            key = (symbol, f'q{q}', year)
            if not (1 <= q <= 4) or key in existing or pub_date <= cutoff:
                n_skip += 1
                continue
            try:
                tdf = tr.get_transcript(year, q)
                # columns: paragraph_number, speaker, content
            except Exception as e:
                n_fail += 1
                print(f'  {symbol} q{q} {year}: fetch failed ({type(e).__name__})')
                continue
            turns = normalize_turns(
                [{'speaker': t.speaker, 'text': t.content}
                 for t in tdf.itertuples()])
            if not turns:
                n_skip += 1
                continue
            sections = split_turns(turns)
            if not sections['full_text']:
                n_skip += 1
                continue
            batch.append(make_row(symbol, f'q{q}', year, pub_date,
                                  'defeatbeta', sections))
            existing.add(key)

        write_batch(conn, batch)   # one transaction per ticker
        n_new += len(batch)
        if (i + 1) % 25 == 0 or batch:
            el = time.time() - t0
            msg = f'+{len(batch)} new' if batch else 'up to date'
            print(f'  [{i+1}/{len(universe)}] {symbol}: {msg} '
                  f'(total new={n_new:,}, {el/60:.1f} min)')

    conn.close()
    print(f'\nUpdate done in {(time.time()-t0)/60:.1f} min: '
          f'{n_new:,} new transcripts, {n_skip:,} already current/skipped, '
          f'{n_fail} failures')
else:
    print('RUN_UPDATE=False — flip to fetch post-2025 quarters via defeatbeta-api')